In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 36.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import transformers
print(transformers.__version__)

5.4.0


In [ ]:
pip install torch datasets scikit-learn pandas openpyxl

In [ ]:
import pandas as pd

df = pd.read_excel("train_manual_label.xlsx")
df = df.dropna(subset=["ulasan_clean", "aspek", "sentiment"])

rows = []
for _, r in df.iterrows():
    aspek = [a.strip() for a in r["aspek"].split(";")]
    sent  = [s.strip() for s in r["sentiment"].split(";")]

    if len(aspek) != len(sent):
        continue

    for a, s in zip(aspek, sent):
        rows.append({
            "text": r["ulasan_clean"] + " [ASPEK] " + a,
            "label": s
        })

df_train = pd.DataFrame(rows)

In [ ]:
label2id = {"negatif": 0, "netral": 1, "positif": 2}
id2label = {v:k for k,v in label2id.items()}

df_train["label"] = df_train["label"].map(label2id)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(
    df_train,
    test_size=0.2,
    random_state=42,
    stratify=df_train["label"]
)

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(
    "indobenchmark/indobert-base-p1"
)

train_ds = Dataset.from_pandas(train_df)
eval_ds  = Dataset.from_pandas(eval_df)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
eval_ds  = eval_ds.map(tokenize, batched=True)

train_ds.set_format("torch", columns=["input_ids","attention_mask","label"])
eval_ds.set_format("torch", columns=["input_ids","attention_mask","label"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/644 [00:00<?, ? examples/s]

Map:   0%|          | 0/162 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p1",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer.train()

You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=243, training_loss=0.34679159117333686, metrics={'train_runtime': 2552.0877, 'train_samples_per_second': 0.757, 'train_steps_per_second': 0.095, 'total_flos': 127083780762624.0, 'train_loss': 0.34679159117333686, 'epoch': 3.0})

In [ ]:
pred = trainer.predict(eval_ds)
y_true = eval_df["label"].values
y_pred = pred.predictions.argmax(axis=1)

from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=label2id.keys()))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

     negatif       0.79      0.65      0.71        17
      netral       1.00      0.18      0.31        11
     positif       0.91      0.99      0.95       134

    accuracy                           0.90       162
   macro avg       0.90      0.61      0.66       162
weighted avg       0.90      0.90      0.88       162



In [ ]:
trainer.save_model("indobert_absa_final")
tokenizer.save_pretrained("indobert_absa_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('indobert_absa_final/tokenizer_config.json',
 'indobert_absa_final/tokenizer.json')

In [ ]:
!zip -r indobert_absa_final.zip indobert_absa_final

from google.colab import files
files.download("indobert_absa_final.zip")

  adding: indobert_absa_final/ (stored 0%)
  adding: indobert_absa_final/tokenizer.json (deflated 71%)
  adding: indobert_absa_final/training_args.bin (deflated 54%)
  adding: indobert_absa_final/config.json (deflated 57%)
  adding: indobert_absa_final/model.safetensors (deflated 7%)
  adding: indobert_absa_final/tokenizer_config.json (deflated 41%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>